# LeetCode 4064 — Longest Subarray Divisible by K with At Most One Negation II (Hard)

Bài toán LeetCode:

```python
class Solution:
    def longestSubarray(self, nums: List[int], k: int) -> int:
        ...
```

Link tra cứu: https://leetcode.com/problemset/?search=4064 (Biweekly Contest 192 — câu 4, rating khoảng 2284).

Bài này dựa trên kỹ thuật kinh điển **tổng tiền tố theo modulo** (prefix sum mod k) — "đoạn có tổng chia hết cho `k`" ⇔ "hai tổng tiền tố có cùng số dư". Cái mới là được **đổi dấu tối đa một phần tử**. Ta sẽ:

- biến "đổi dấu `x`" thành một điều kiện số dư: `p[r] ≡ p[l] + 2x (mod k)`;
- nhận ra chỉ cần **lần xuất hiện sớm nhất** của mỗi số dư;
- khai thác `k <= 3000` nhỏ để đạt `O(n + k²)`.

---

## Sau notebook này, bạn sẽ nắm được

1. Tổng tiền tố modulo, phép `%` với số âm trong Python.
2. Brute Force `O(n³)` và `O(n²)`.
3. Điều kiện số dư khi đổi dấu một phần tử.
4. Cách `O(n · k)`: thử mọi số dư của điểm đầu, dùng "vị trí gần nhất" của phần tử cần đổi dấu.
5. Cách `O(n + k²)`: mỗi cặp (số dư điểm đầu, số dư phần tử) chỉ xử lý **một lần**.
6. Lỗi hay gặp, mẹo nhận diện, biến thể và bài tập.

# 1. Đề bài yêu cầu gì?

Cho mảng số nguyên `nums` (có thể âm) và số nguyên `k`. Một đoạn con (liên tiếp) được gọi là **hợp lệ** nếu:

- tổng của nó chia hết cho `k`, **hoặc**
- **đổi dấu một phần tử** trong đoạn (thay `x` bằng `-x`) thì tổng chia hết cho `k`.

Trả về độ dài đoạn hợp lệ dài nhất, hoặc `0` nếu không có.

---

## Ví dụ 1

```python
nums = [4, 1, 2], k = 3   ->   3
```

Tổng `7` không chia hết cho 3. Đổi dấu `2`: `4 + 1 - 2 = 3` ✔ → cả mảng hợp lệ.

## Ví dụ 2

```python
nums = [5, 3, 4], k = 7   ->   2
```

Cả mảng (tổng 12) không hợp lệ dù đổi dấu phần tử nào. `[3, 4]` có tổng 7 ✔.

## Ví dụ 3

```python
nums = [2, 2, 5], k = 6   ->   2
```

`[2, 2]`: tổng 4, đổi dấu một số 2 → `0` ✔.

---

## Ràng buộc

```text
1 <= nums.length <= 10^5
-10^5 <= nums[i] <= 10^5
1 <= k <= 3000
```

# 2. Kiến thức nền tảng

## 2.1. Phép `%` với số âm trong Python

Python luôn trả về số dư **không âm** khi `k > 0`: `-7 % 3 == 2`. (Trong C++/Java, `-7 % 3 == -1` — phải tự cộng `k`.)

In [1]:
print(-7 % 3, 7 % 3, -6 % 3, (-1) % 5)

2 1 0 4


## 2.2. Tổng tiền tố modulo

`p[0] = 0`, `p[i+1] = (p[i] + nums[i]) % k`. Tổng của đoạn `nums[l..r-1]` là `S = P[r] - P[l]` (tổng tiền tố thật), nên:

```text
S ≡ 0 (mod k)   ⇔   p[r] == p[l]
```

Đoạn dài nhất có tổng chia hết cho `k`: với mỗi `r`, lấy `l` **sớm nhất** có `p[l] == p[r]`.

In [2]:
from typing import List

nums, k = [4, 1, 2], 3
p = [0]
for x in nums:
    p.append((p[-1] + x) % k)
print("p =", p)

p = [0, 1, 2, 1]


## 2.3. Đổi dấu một phần tử

Đổi `x` thành `-x` làm tổng giảm `2x`. Đoạn `[l, r)` chứa phần tử `x` (ở vị trí `t`, `l <= t < r`) trở thành chia hết cho `k` khi:

```text
S - 2x ≡ 0   ⇔   p[r] - p[l] ≡ 2x   ⇔   p[l] ≡ p[r] - 2x   (mod k)
```

Chỉ cần biết `x mod k`. Ví dụ 1: `S = 7`, `x = 2`: `7 - 4 = 3 ≡ 0` ✔.

# 3. Cách 1 — Brute Force O(n³)

In [3]:
def longest_bruteforce(nums: List[int], k: int) -> int:
    n = len(nums)
    best = 0
    for l in range(n):
        for r in range(l, n):
            seg = nums[l:r + 1]
            S = sum(seg)
            if S % k == 0 or any((S - 2 * x) % k == 0 for x in seg):
                best = max(best, r - l + 1)
    return best


print(longest_bruteforce([4, 1, 2], 3), longest_bruteforce([5, 3, 4], 7), longest_bruteforce([2, 2, 5], 6))

3 2 2


# 4. Cách 2 — O(n²): cố định điểm đầu, giữ tập số dư `2x`

Kéo `r` sang phải, giữ tổng modulo `s` và **tập** `twos` các giá trị `2x mod k` của phần tử trong đoạn. Đoạn hợp lệ ⇔ `s == 0` hoặc `s ∈ twos`.

In [4]:
def longest_quadratic(nums: List[int], k: int) -> int:
    n = len(nums)
    best = 0
    for l in range(n):
        s = 0
        twos = set()
        for r in range(l, n):
            s = (s + nums[r]) % k
            twos.add(2 * nums[r] % k)
            if s == 0 or s in twos:
                best = max(best, r - l + 1)
    return best


print(longest_quadratic([4, 1, 2], 3), longest_quadratic([5, 3, 4], 7), longest_quadratic([2, 2, 5], 6))

3 2 2


`O(n²) = 10^10` — quá chậm.

---

# 5. Bước đột phá: chỉ cần **điểm đầu sớm nhất** của mỗi số dư

Cố định điểm cuối `r` (tiền tố `p[r] = s`). Ta muốn `l` nhỏ nhất sao cho đoạn `[l, r)` hợp lệ.

Nhóm các điểm đầu `l` theo số dư `q = p[l]`. Trong một nhóm, chọn `l = first[q]` (lần xuất hiện **sớm nhất**) luôn tốt nhất:

- đoạn dài nhất;
- điều kiện "có phần tử `x` ở vị trí `t ∈ [l, r)` với `2x ≡ s - q`" **dễ thỏa nhất** khi `l` nhỏ nhất (khoảng `[l, r)` rộng nhất).

Vậy chỉ có **`k` ứng viên** điểm đầu: `first[0], ..., first[k-1]`.

# 6. Cách 3 — O(n · k)

Với mỗi `r`, thử mọi `q`:

- `q == s` → hợp lệ không cần đổi dấu;
- hoặc cần một phần tử ở vị trí `t ∈ [first[q], r)` với `2x ≡ s - q`. Giữ `lastpos[v]` = vị trí **gần nhất** (`< r`) của phần tử có `2x mod k == v`. Điều kiện: `lastpos[(s - q) % k] >= first[q]`.

In [5]:
def longest_nk(nums: List[int], k: int) -> int:
    n = len(nums)
    first = [-1] * k
    first[0] = 0
    lastpos = [-1] * k
    s = 0
    best = 0
    for i in range(n):
        lastpos[2 * nums[i] % k] = i          # phần tử i có thể là phần tử bị đổi dấu
        s = (s + nums[i]) % k                 # s = p[i+1]
        r = i + 1
        if first[s] == -1:
            first[s] = r
        for q in range(k):
            if first[q] == -1 or first[q] >= r:
                continue
            if q == s or lastpos[(s - q) % k] >= first[q]:
                best = max(best, r - first[q])
    return best


print(longest_nk([4, 1, 2], 3), longest_nk([5, 3, 4], 7), longest_nk([2, 2, 5], 6))

3 2 2


`n · k = 3·10^8` — vẫn quá chậm trong Python. Còn lãng phí gì?

# 7. Bước đột phá 2: mỗi cặp (số dư điểm đầu, số dư phần tử) chỉ cần **một lần**

Đảo góc nhìn: thay vì với mỗi `r` hỏi "điểm đầu nào được?", ta **chuẩn bị trước** cho mọi số dư đích `t`:

```text
best_l[t] = điểm đầu nhỏ nhất l sao cho: mọi r > (vị trí phần tử dùng để đổi dấu) có p[r] = t
            đều làm đoạn [l, r) hợp lệ
```

Khởi tạo `best_l[t] = first[t]` (không đổi dấu).

Khi duyệt tới phần tử `i` với `a = nums[i] mod k`: với mọi số dư `q` mà `first[q] <= i` (điểm đầu đứng trước hoặc tại `i`), đổi dấu phần tử `i` giúp các điểm cuối `r` có `p[r] ≡ q + 2a` dùng được điểm đầu `first[q]`:

```text
best_l[(q + 2a) % k] = min(best_l[...], first[q])
```

Rồi với `r = i + 1`, `s = p[r]`: ứng viên độ dài `r - best_l[s]`.

**Mấu chốt:** cặp `(a, q)` chỉ cần xử lý **lần đầu tiên** gặp một phần tử có số dư `a` ở vị trí `>= first[q]` — các phần tử sau cùng số dư `a` cho đúng cùng cập nhật. Sắp các số dư `q` theo `first[q]` tăng dần (`order`), mỗi số dư `a` giữ một con trỏ `ptr[a]` chỉ tiến → tổng `O(k²)` cập nhật.

In [6]:
def longest_fast(nums: List[int], k: int, verbose: bool = False) -> int:
    n = len(nums)
    p = [0] * (n + 1)
    for i, x in enumerate(nums):
        p[i + 1] = (p[i] + x) % k
    INF = n + 1
    first = [INF] * k
    for idx in range(n + 1):
        if first[p[idx]] == INF:
            first[p[idx]] = idx
    order = sorted((q for q in range(k) if first[q] < INF), key=lambda q: first[q])
    best_l = first[:]                          # không đổi dấu
    ptr = [0] * k
    ans = 0
    for i in range(n):
        a = nums[i] % k
        while ptr[a] < len(order) and first[order[ptr[a]]] <= i:
            q = order[ptr[a]]
            t = (q + 2 * a) % k
            if first[q] < best_l[t]:
                best_l[t] = first[q]
            ptr[a] += 1
        s = p[i + 1]
        if best_l[s] <= i:                     # đoạn [best_l[s], i+1) khác rỗng
            ans = max(ans, i + 1 - best_l[s])
        if verbose:
            print(f"i = {i}, a = {a}, s = p[{i + 1}] = {s}, best_l = {best_l}, ans = {ans}")
    return ans


print(longest_fast([4, 1, 2], 3, verbose=True))

i = 0, a = 1, s = p[1] = 1, best_l = [0, 1, 0], ans = 0
i = 1, a = 1, s = p[2] = 2, best_l = [0, 1, 0], ans = 2
i = 2, a = 2, s = p[3] = 1, best_l = [0, 0, 0], ans = 3
3


## Mô phỏng ví dụ 1 (`nums = [4, 1, 2]`, `k = 3`)

`p = [0, 1, 2, 1]`, `first = {0: 0, 1: 1, 2: 2}`, `order = [0, 1, 2]`, `best_l = [0, 1, 2]` ban đầu.

| i | a | cặp (a, q) mới xử lý | cập nhật | s = p[i+1] | ứng viên |
|---:|---:|---|---|---:|---|
| 0 | 1 | `q = 0` | `best_l[(0+2)%3 = 2] = 0` | 1 | `best_l[1] = 1 > 0` → không |
| 1 | 1 | `q = 1` | `best_l[(1+2)%3 = 0] = 0` (đã là 0) | 2 | `2 - best_l[2] = 2 - 0 = 2` |
| 2 | 2 | `q = 0, 1, 2` | `best_l[1] = 0`, ... | 1 | `3 - 0 = 3` |

Đáp án `3` ✔. Ở `i = 2`, cặp `(a = 2, q = 0)` nghĩa là: điểm đầu `0`, đổi dấu số `2` → các điểm cuối có `p[r] ≡ 0 + 4 ≡ 1` hợp lệ; `p[3] = 1` ✔.

# 8. Phiên bản đúng format LeetCode

In [7]:
from typing import List


class Solution:
    def longestSubarray(self, nums: List[int], k: int) -> int:
        n = len(nums)
        p = [0] * (n + 1)
        for i, x in enumerate(nums):
            p[i + 1] = (p[i] + x) % k
        INF = n + 1
        first = [INF] * k
        for idx in range(n + 1):
            if first[p[idx]] == INF:
                first[p[idx]] = idx
        order = sorted((q for q in range(k) if first[q] < INF), key=lambda q: first[q])
        best_l = first[:]
        ptr = [0] * k
        ans = 0
        m = len(order)
        for i in range(n):
            a = nums[i] % k
            pa = ptr[a]
            while pa < m and first[order[pa]] <= i:
                q = order[pa]
                t = (q + 2 * a) % k
                if first[q] < best_l[t]:
                    best_l[t] = first[q]
                pa += 1
            ptr[a] = pa
            bl = best_l[p[i + 1]]
            if bl <= i and i + 1 - bl > ans:
                ans = i + 1 - bl
        return ans

In [8]:
sol = Solution()
tests = [
    ([4, 1, 2], 3, 3, "ví dụ 1"),
    ([5, 3, 4], 7, 2, "ví dụ 2"),
    ([2, 2, 5], 6, 2, "ví dụ 3"),
    ([1], 1, 1, "k = 1: mọi đoạn hợp lệ"),
    ([1], 2, 0, "k = 2, phần tử lẻ: đổi dấu vẫn lẻ"),
    ([-3, 5], 4, 2, "số âm"),
    ([1, 1, 1], 7, 2, "[1, 1]: đổi dấu một số 1 được tổng 0"),
]
for nums, k, expected, name in tests:
    result = sol.longestSubarray(nums, k)
    assert result == expected, (name, result, expected)
    print(f"PASS | {name:44} -> {result}")

PASS | ví dụ 1                                      -> 3
PASS | ví dụ 2                                      -> 2
PASS | ví dụ 3                                      -> 2
PASS | k = 1: mọi đoạn hợp lệ                       -> 1
PASS | k = 2, phần tử lẻ: đổi dấu vẫn lẻ            -> 0
PASS | số âm                                        -> 2
PASS | [1, 1]: đổi dấu một số 1 được tổng 0         -> 2


# 9. Vì sao đúng?

1. **Điều kiện số dư:** đoạn `[l, r)` hợp lệ ⇔ `p[l] == p[r]` hoặc tồn tại `t ∈ [l, r)` với `p[l] ≡ p[r] - 2·nums[t]`.
2. **Điểm đầu sớm nhất:** với cùng số dư `q`, `first[q]` vừa cho đoạn dài nhất vừa làm điều kiện "tồn tại `t >= l`" dễ thỏa nhất.
3. **`best_l[t]` đúng tại thời điểm `r = i + 1`:** nó là min của `first[t]` và mọi `first[q]` mà đã có một phần tử ở vị trí `j ∈ [first[q], i]` với `q + 2·nums[j] ≡ t`. Đó đúng là tập các điểm đầu hợp lệ cho điểm cuối `r` có `p[r] = t` (vì `j < r`).
4. **Không bỏ sót cặp:** cặp `(a, q)` được xử lý ngay khi gặp phần tử đầu tiên có số dư `a` ở vị trí `>= first[q]`; các phần tử sau cùng số dư `a` không thêm gì mới.

# 10. Độ phức tạp

| Cách | Ý tưởng | Thời gian | Bộ nhớ |
|---|---|---:|---:|
| 1. Brute Force | mọi đoạn, thử mọi phần tử | `O(n³)` | `O(n)` |
| 2. Cố định điểm đầu | tập số dư `2x` | `O(n²)` | `O(k)` |
| 3. Thử mọi số dư điểm đầu | `first`, `lastpos` | `O(n · k)` | `O(k)` |
| 4. Mỗi cặp một lần | `order` + con trỏ | `O(n + k²)` | `O(n + k)` |

Với `k = 3000`: `k² = 9·10^6`, chấp nhận được.

# 11. Đối chiếu bốn cách

In [9]:
import random
import time

random.seed(4064)
for _ in range(3000):
    k = random.randint(1, 8)
    nums = [random.randint(-9, 9) for _ in range(random.randint(1, 9))]
    expected = longest_bruteforce(nums, k)
    assert longest_quadratic(nums, k) == expected, (nums, k)
    assert longest_nk(nums, k) == expected, (nums, k)
    assert longest_fast(nums, k) == expected, (nums, k)
    assert sol.longestSubarray(nums, k) == expected, (nums, k)
print("3000 bộ test ngẫu nhiên: 5 hàm cho cùng kết quả!")

big = [random.randint(-10**5, 10**5) for _ in range(10**5)]
t0 = time.perf_counter()
print("n = 10^5, k = 3000:", sol.longestSubarray(big, 3000), f"({time.perf_counter() - t0:.2f}s)")

3000 bộ test ngẫu nhiên: 5 hàm cho cùng kết quả!


n = 10^5, k = 3000: 99999 (1.43s)


# 12. Những lỗi dễ mắc

## Lỗi 1 — Nghĩ đổi dấu làm tổng thay đổi `x` (thay vì `2x`)

Thay `x` bằng `-x`: tổng mới = `S - x - x = S - 2x`.

In [10]:
def wrong_minus_x(nums, k):
    n, best = len(nums), 0
    for l in range(n):
        for r in range(l, n):
            seg = nums[l:r + 1]
            S = sum(seg)
            if S % k == 0 or any((S - x) % k == 0 for x in seg):      # SAI: S - x
                best = max(best, r - l + 1)
    return best


print("Sai :", wrong_minus_x([5, 3, 4], 7))
print("Đúng:", sol.longestSubarray([5, 3, 4], 7))

Sai : 3
Đúng: 2


## Lỗi 2 — Cho phép phần tử đổi dấu nằm **ngoài** đoạn

Điều kiện `t >= l` (phần tử nằm trong đoạn) là bắt buộc. Nếu chỉ kiểm tra "đã từng có phần tử với `2x ≡ s - q`" mà không so với `first[q]`, ta dùng nhầm phần tử nằm trước điểm đầu.

In [11]:
def wrong_outside(nums, k):
    n = len(nums)
    first = [-1] * k
    first[0] = 0
    seen2 = set()
    s = best = 0
    for i in range(n):
        seen2.add(2 * nums[i] % k)
        s = (s + nums[i]) % k
        if first[s] == -1:
            first[s] = i + 1
        for q in range(k):
            if first[q] != -1 and first[q] <= i and (q == s or (s - q) % k in seen2):   # SAI
                best = max(best, i + 1 - first[q])
    return best


random.seed(2)
for _ in range(5000):
    k = random.randint(2, 7)
    arr = [random.randint(-6, 6) for _ in range(random.randint(1, 7))]
    if wrong_outside(arr, k) != sol.longestSubarray(arr, k):
        print("Phản ví dụ:", arr, k, "| sai:", wrong_outside(arr, k), "| đúng:", sol.longestSubarray(arr, k))
        break

Phản ví dụ: [-1, 0, 0, 2] 4 | sai: 3 | đúng: 2


## Lỗi 3 — Quên `p[0] = 0` (điểm đầu ở chỉ số 0)

Nếu không ghi nhận `first[0] = 0`, các đoạn bắt đầu từ đầu mảng bị bỏ sót.

## Lỗi 4 — Số dư âm trong C++/Java

`nums[i]` âm → `nums[i] % k` âm trong C++/Java. Phải chuẩn hóa `((x % k) + k) % k`. Python không cần.

## Lỗi 5 — Đếm đoạn rỗng

`best_l[s]` có thể bằng `i + 1` (chính `first[s] = r`) → độ dài 0. Kiểm tra `best_l[s] <= i`.

# 13. Mẹo nhận diện

1. "Đoạn có tổng chia hết cho `k`" → **tổng tiền tố modulo**, `p[l] == p[r]`, lấy `first[...]` cho đoạn dài nhất.
2. "Được sửa tối đa một phần tử" → biến sự sửa đổi thành một **dịch chuyển số dư** (`+2x`, `-x`, ...) và thêm điều kiện "phần tử nằm trong đoạn".
3. `k` nhỏ (≤ vài nghìn) cho phép `O(k²)` — thử nghĩ "mỗi cặp số dư xử lý một lần".
4. Điều kiện "tồn tại `t` trong `[l, r)`" với `l` sớm nhất → chỉ cần lần xuất hiện **gần nhất** (hoặc "đã xuất hiện sau `l`").

Các bài liên quan: *Subarray Sums Divisible by K* (LeetCode 974), *Continuous Subarray Sum* (523), *Make Sum Divisible by P* (1590), *Longest Subarray Divisible by K with At Most One Negation I*.

# 14. Biến thể

## Biến thể 1 — Không được đổi dấu (đoạn dài nhất có tổng chia hết cho `k`)

In [12]:
def longest_divisible(nums: List[int], k: int) -> int:
    first = {0: 0}
    s = best = 0
    for i, x in enumerate(nums):
        s = (s + x) % k
        if s in first:
            best = max(best, i + 1 - first[s])
        else:
            first[s] = i + 1
    return best


print(longest_divisible([5, 3, 4], 7), longest_divisible([4, 1, 2], 3))

2 2


## Biến thể 2 — Được **xóa** tối đa một phần tử

Xóa `x` làm tổng giảm `x` (không phải `2x`): chỉ cần thay `2a` bằng `a` trong cách 4. (Bài tập 5.)

# 15. Template tư duy

```python
p = prefix mod k; first[q] = vị trí sớm nhất của số dư q
best_l = first[:]                                  # không sửa
order = các q sắp theo first
for i, x in enumerate(nums):
    a = x % k
    while ptr[a] chưa hết và first[order[ptr[a]]] <= i:
        q = order[ptr[a]]; best_l[(q + shift(a)) % k] = min(..., first[q]); ptr[a] += 1
    if best_l[p[i+1]] <= i: ans = max(ans, i + 1 - best_l[p[i+1]])
```

# 16. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Ví dụ 3 (`nums = [2, 2, 5]`, `k = 6`): tính `p`, `first`, rồi mô phỏng cách 4. Kiểm tra bằng cell dưới.

In [13]:
# Kiểm tra Bài 1
print(longest_fast([2, 2, 5], 6, verbose=True))

i = 0, a = 2, s = p[1] = 2, best_l = [0, 4, 1, 3, 0, 4], ans = 0
i = 1, a = 2, s = p[2] = 4, best_l = [0, 4, 1, 3, 0, 4], ans = 2
i = 2, a = 5, s = p[3] = 3, best_l = [0, 4, 1, 3, 0, 4], ans = 2
2


## Bài 2 — Dự đoán output

```python
print(-5 % 6, (2 * -5) % 6, (0 + 2 * 2) % 6)
```

In [14]:
print(-5 % 6, (2 * -5) % 6, (0 + 2 * 2) % 6)

1 2 4


## Bài 3 — Hoàn thiện cách O(n²)

In [15]:
def exercise_3_quadratic(nums: List[int], k: int) -> int:
    # TODO: cố định l, kéo r; giữ s = tổng mod k và tập twos = {2x mod k}
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")


# assert exercise_3_quadratic([4, 1, 2], 3) == 3

## Bài 4 — Subarray Sums Divisible by K (LeetCode 974)

Đếm số đoạn có tổng chia hết cho `k`.

```python
count_divisible([4, 5, 0, -2, -3, 1], 5)   -> 7
```

In [16]:
def exercise_4_count_divisible(nums: List[int], k: int) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_4_count_divisible([4, 5, 0, -2, -3, 1], 5) == 7

## Bài 5 — Biến thể: được **xóa** tối đa một phần tử

Đoạn hợp lệ nếu tổng chia hết cho `k`, hoặc xóa một phần tử trong đoạn thì chia hết (đoạn sau khi xóa có thể rỗng — tính độ dài **trước** khi xóa). Viết lời giải `O(n + k²)` và kiểm tra với brute force.

In [17]:
def exercise_5_with_deletion(nums: List[int], k: int) -> int:
    # TODO
    raise NotImplementedError

## Bài 6 — Mini challenge: chỉ ra đoạn và phần tử đổi dấu

Trả về `(l, r, t)`: đoạn `nums[l..r]` dài nhất và chỉ số `t` của phần tử cần đổi dấu (hoặc `-1` nếu không cần). Gợi ý: cùng với `best_l`, lưu chỉ số phần tử đã tạo ra giá trị đó.

In [18]:
def exercise_6_witness(nums: List[int], k: int):
    # TODO
    raise NotImplementedError

## Gợi ý (mở từng mức)

**Bài 3**

- Mức 1: `s = (s + nums[r]) % k`, `twos.add(2 * nums[r] % k)`.
- Mức 2: hợp lệ khi `s == 0 or s in twos`.
- Mức 3: cập nhật `best = max(best, r - l + 1)`.

**Bài 4**

- Mức 1: đếm số lần mỗi số dư tiền tố xuất hiện (bắt đầu `{0: 1}`).
- Mức 2: với mỗi tiền tố mới `s`, cộng `cnt[s]` (số điểm đầu cùng số dư).
- Mức 3: rồi `cnt[s] += 1`.

**Bài 5**

- Mức 1: xóa `x` → tổng giảm `x`: điều kiện `p[l] ≡ p[r] - x`.
- Mức 2: thay `t = (q + 2a) % k` bằng `t = (q + a) % k`.
- Mức 3: brute force: thử mọi đoạn và mọi phần tử bị xóa.

**Bài 6**

- Mức 1: `who[t]` = `(first[q], j)` với `j` là phần tử đổi dấu khi cập nhật `best_l[t]`.
- Mức 2: khởi tạo `who[t] = (first[t], -1)`.
- Mức 3: khi cập nhật đáp án, ghi lại `(best_l[s], i, who[s][1])`.

# 17. Đáp án tham khảo

## Đáp án Bài 1

`p = [0, 2, 4, 3]`, `first = {0: 0, 2: 1, 4: 2, 3: 3}`. Ngay ở `i = 0` (`a = 2`), cặp `(2, q = 0)` cho `best_l[4] = 0`; tới `i = 1`, `p[2] = 4` → đoạn `[0, 2)` dài 2 (đổi dấu một số 2). Không có đoạn dài 3 hợp lệ. Đáp án `2`.

## Đáp án Bài 2

```text
1 2 4
```

In [19]:
# Đáp án Bài 3
def answer_3_quadratic(nums: List[int], k: int) -> int:
    n, best = len(nums), 0
    for l in range(n):
        s, twos = 0, set()
        for r in range(l, n):
            s = (s + nums[r]) % k
            twos.add(2 * nums[r] % k)
            if s == 0 or s in twos:
                best = max(best, r - l + 1)
    return best


assert answer_3_quadratic([4, 1, 2], 3) == 3
print("Bài 3: PASS")

Bài 3: PASS


In [20]:
# Đáp án Bài 4
def answer_4_count_divisible(nums: List[int], k: int) -> int:
    cnt = {0: 1}
    s = total = 0
    for x in nums:
        s = (s + x) % k
        total += cnt.get(s, 0)
        cnt[s] = cnt.get(s, 0) + 1
    return total


assert answer_4_count_divisible([4, 5, 0, -2, -3, 1], 5) == 7
print("Bài 4: PASS")

Bài 4: PASS


In [21]:
# Đáp án Bài 5
def answer_5_with_deletion(nums: List[int], k: int) -> int:
    n = len(nums)
    p = [0] * (n + 1)
    for i, x in enumerate(nums):
        p[i + 1] = (p[i] + x) % k
    INF = n + 1
    first = [INF] * k
    for idx in range(n + 1):
        if first[p[idx]] == INF:
            first[p[idx]] = idx
    order = sorted((q for q in range(k) if first[q] < INF), key=lambda q: first[q])
    best_l, ptr, ans = first[:], [0] * k, 0
    for i in range(n):
        a = nums[i] % k
        while ptr[a] < len(order) and first[order[ptr[a]]] <= i:
            q = order[ptr[a]]
            t = (q + a) % k
            best_l[t] = min(best_l[t], first[q])
            ptr[a] += 1
        if best_l[p[i + 1]] <= i:
            ans = max(ans, i + 1 - best_l[p[i + 1]])
    return ans


def brute_deletion(nums, k):
    n, best = len(nums), 0
    for l in range(n):
        for r in range(l, n):
            seg = nums[l:r + 1]
            S = sum(seg)
            if S % k == 0 or any((S - x) % k == 0 for x in seg):
                best = max(best, r - l + 1)
    return best


random.seed(5)
for _ in range(2000):
    k = random.randint(1, 8)
    arr = [random.randint(-9, 9) for _ in range(random.randint(1, 9))]
    assert answer_5_with_deletion(arr, k) == brute_deletion(arr, k), (arr, k)
print("Bài 5: PASS (khớp brute force trên 2000 test)")

Bài 5: PASS (khớp brute force trên 2000 test)


In [22]:
# Đáp án Bài 6
def answer_6_witness(nums: List[int], k: int):
    n = len(nums)
    p = [0] * (n + 1)
    for i, x in enumerate(nums):
        p[i + 1] = (p[i] + x) % k
    INF = n + 1
    first = [INF] * k
    for idx in range(n + 1):
        if first[p[idx]] == INF:
            first[p[idx]] = idx
    order = sorted((q for q in range(k) if first[q] < INF), key=lambda q: first[q])
    best_l, who, ptr = first[:], [-1] * k, [0] * k
    ans, res = 0, None
    for i in range(n):
        a = nums[i] % k
        while ptr[a] < len(order) and first[order[ptr[a]]] <= i:
            q = order[ptr[a]]
            t = (q + 2 * a) % k
            if first[q] < best_l[t]:
                best_l[t], who[t] = first[q], i
            ptr[a] += 1
        s = p[i + 1]
        if best_l[s] <= i and i + 1 - best_l[s] > ans:
            ans, res = i + 1 - best_l[s], (best_l[s], i, who[s])
    return res


random.seed(6)
for _ in range(2000):
    k = random.randint(1, 8)
    arr = [random.randint(-9, 9) for _ in range(random.randint(1, 9))]
    res = answer_6_witness(arr, k)
    expected = longest_bruteforce(arr, k)
    if expected == 0:
        assert res is None
        continue
    l, r, t = res
    seg = arr[l:r + 1]
    S = sum(seg) - (2 * arr[t] if t != -1 else 0)
    assert r - l + 1 == expected and S % k == 0 and (t == -1 or l <= t <= r), (arr, k, res)
print("Bài 6: PASS | ví dụ 1:", answer_6_witness([4, 1, 2], 3))

Bài 6: PASS | ví dụ 1: (0, 2, 2)


# 18. Tổng kết

**Longest Subarray Divisible by K with At Most One Negation II**:

1. Tổng tiền tố modulo; đổi dấu `x` ⇔ `p[l] ≡ p[r] - 2x`.
2. Mỗi số dư điểm đầu chỉ cần `first[q]`.
3. `best_l[t]` = điểm đầu tốt nhất cho số dư điểm cuối `t`; mỗi cặp (số dư phần tử, số dư điểm đầu) cập nhật một lần nhờ con trỏ trên `order`.

## Độ phức tạp

```text
Time:  O(n + k²)
Space: O(n + k)
```

# 19. Checklist trước khi submit

- [ ] Đổi dấu làm tổng giảm **`2x`**?
- [ ] Phần tử đổi dấu nằm **trong** đoạn (`t >= l`)?
- [ ] Có `p[0] = 0` (điểm đầu ở chỉ số 0)?
- [ ] Số dư âm được chuẩn hóa (C++/Java)?
- [ ] Không đếm đoạn rỗng (`best_l[s] <= i`)?
- [ ] Độ phức tạp `O(n + k²)`, không `O(n · k)`?